# Prompt caching — measure the savings

Prompt caching reuses a large, stable prompt **prefix** across requests: you pay a small write
premium once, then later requests read the prefix back at ~0.1x input price (and faster). It's a
**prefix match** — any byte change before a `cache_control` breakpoint invalidates everything
after it. Render order is `tools` → `system` → `messages`, so one breakpoint on the last system
block caches the tools **and** the system prompt together.

Here we cache a sizable employee handbook (+ system instructions + a tool), then watch the
`usage` metrics prove it: a **write** on the first call, **reads** on the rest, and the classic
**silent invalidator** that breaks it.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root).

## Setup

Helpers live in `_prompt_caching.py`: the generated handbook, the prompt assembly (`build_system`/`build_tools`), and `ask` / `usage_of` / `relative_input_cost`.

In [ ]:
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

for _p in (".", "prompt_caching"):
    if os.path.isfile(os.path.join(_p, "_prompt_caching.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _prompt_caching import (
    QUESTIONS,
    ask,
    prefix_token_count,
    relative_input_cost,
    usage_of,
)

load_dotenv()
client = Anthropic()

## The cacheable prefix

For caching to trigger, the prefix must exceed a **model-dependent minimum**: 1024 tokens on
Sonnet 4.5, but 4096 on Opus 4.8 (a 3K-token prefix caches on Sonnet but *silently* won't on
Opus — `cache_creation_input_tokens` just stays 0). Our handbook prefix clears 4096.

In [ ]:
print("cacheable prefix tokens:", prefix_token_count(client))

## Baseline → write → read

Three calls tell the whole story. Watch the `usage`:
- **Baseline** (caching off): the whole prefix is billed as `input` at full price.
- **Cached call #1**: the prefix moves to `cache_write` (~1.25x, paid once).
- **Cached call #2** (a *different* question, same prefix): the prefix is served as `cache_read`
  (~0.1x) and `input` collapses to just the new question.

In [ ]:
r0, t0 = ask(client, QUESTIONS[0], cache=False)
print("baseline    ", usage_of(r0), f"{t0:.2f}s")

r1, t1 = ask(client, QUESTIONS[0], cache=True)
print("cached #1   ", usage_of(r1), f"{t1:.2f}s   <- cache_write")

r2, t2 = ask(client, QUESTIONS[1], cache=True)
print("cached #2   ", usage_of(r2), f"{t2:.2f}s   <- cache_read")

## The savings across many questions

The first cached request writes; every subsequent one reads. `relative_input_cost` compares the
effective input cost to an uncached request of the same prompt (1.0 = no savings).

In [ ]:
print(f"{'question':<48} {'input':>6} {'write':>6} {'read':>6} {'relcost':>8}")
for q in QUESTIONS:
    r, _ = ask(client, q, cache=True)
    u = usage_of(r)
    print(f"{q[:46]:<48} {u['input']:>6} {u['cache_write']:>6} {u['cache_read']:>6} "
          f"{relative_input_cost(u):>8.2f}")

## The silent invalidator

The #1 caching bug: putting something that changes every request — a timestamp, a UUID, a
user id — *before* the breakpoint. The prefix bytes differ, so the cache never hits. Here we
prepend a timestamp and watch `cache_read` fall back to 0 while `cache_write` happens all over
again.

In [ ]:
r, _ = ask(client, QUESTIONS[1], cache=True,
           volatile_prefix="Request time: 2026-06-06T12:00:00Z")
print("with volatile prefix:", usage_of(r), "  <- cache_read is 0; we paid to write again")

## Notes

- **Verify with `usage`.** `cache_creation_input_tokens` = written this request (~1.25x for the
  default 5-min TTL); `cache_read_input_tokens` = served from cache (~0.1x); `input_tokens` = the
  full-price remainder. Total prompt = the sum of all three.
- **Minimum prefix is model-dependent** (1024 Sonnet 4.5 / 2048 Sonnet 4.6 / 4096 Opus & Haiku
  4.5). Below it, caching silently no-ops.
- **Economics.** Reads ~0.1x, writes 1.25x (5-min) or 2x (1-hour, `{"type":"ephemeral","ttl":"1h"}`).
  With the 5-min TTL you break even at ~2 requests.
- **Keep the prefix frozen.** Don't interpolate dates/IDs/flags into `system` or reorder `tools`
  — render order is `tools` → `system` → `messages`, and any earlier byte change invalidates
  what follows. Put volatile content at the *end*, after the last breakpoint.
- **Up to 4 breakpoints** per request. A breakpoint on the last system block caches tools+system
  together (as we do here).
- **Composes with other features** — citations, batch, token counting. (Toggling citations/web
  search invalidates the system cache tier but not tools.)